In [0]:
%sql
-- Considerando que Ativação é ao menos 1 transação no dia
-- DAU
SELECT 
    SUBSTR(DtCriacao, 0, 11) AS DtDay,
    DATE(DtCriacao) AS dtDia,
    count(*) AS qtdeTransacoes,
    COUNT(DISTINCT IdCliente) AS DAU
FROM workspace.tmw_loyalty.transacoes
GROUP BY ALL
ORDER BY dtDia

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Considerando que Ativação é ao menos 1 transação no dia
-- MAU (D28)
WITH 
tb_daily AS (
    SELECT DISTINCT
        DATE(SUBSTR(DtCriacao, 0, 11)) AS DtDay,
        IdCliente
    FROM tmw_loyalty.transacoes
    ORDER BY DtDay
),
tb_distinct_days AS (
    SELECT
        DISTINCT DtDay AS DtRef
    FROM tb_daily
)
SELECT
    t1.DtRef,
    COUNT(DISTINCT t2.IdCliente) AS MAU,
    COUNT(DISTINCT t2.DtDay) AS DaysCount
FROM tb_distinct_days AS t1
LEFT JOIN tb_daily AS t2
    ON t2.DtDay <= t1.DtRef
    -- AND julianday(t1.DtRef) - julianday(t2.DtDay) < 28
    AND datediff(t1.DtRef, t2.DtDay) < 28
GROUP BY t1.DtRef
ORDER BY DtRef ASC ;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Considerando que Ativação é ao menos 1 transação no dia
-- WAU
WITH 
tb_daily AS (
    SELECT DISTINCT
        DATE(SUBSTR(DtCriacao, 0, 11)) AS DtDay,
        IdCliente
    FROM tmw_loyalty.transacoes
    ORDER BY DtDay
),
tb_distinct_days AS (
    SELECT
        DISTINCT DtDay AS DtRef
    FROM tb_daily
)
SELECT
    t1.DtRef,
    COUNT(DISTINCT t2.IdCliente) AS WAU,
    COUNT(DISTINCT t2.DtDay) AS DaysCount
FROM tb_distinct_days AS t1
LEFT JOIN tb_daily AS t2
    ON t2.DtDay <= t1.DtRef
    -- AND julianday(t1.DtRef) - julianday(t2.DtDay) < 7
    AND datediff(t1.DtRef, t2.DtDay) < 7
GROUP BY t1.DtRef
ORDER BY DtRef ASC ;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Considerando que Ativação é ao menos 1 transação no dia
-- ARPU -> Considerando D28 e apenas transações positivas
WITH 
tb_daily AS (
    SELECT DISTINCT
        DATE(SUBSTR(DtCriacao, 0, 11)) AS DtDay,
        IdCliente,
        (CASE WHEN QtdePontos > 0 THEN QtdePontos ELSE 0 END) AS QtdePontos_pos,
        (CASE WHEN QtdePontos < 0 THEN QtdePontos ELSE 0 END) AS QtdePontos_neg
    FROM tmw_loyalty.transacoes
    ORDER BY DtDay
),
tb_distinct_days AS (
    SELECT
        DISTINCT DtDay AS DtRef
    FROM tb_daily
)
SELECT
    t1.DtRef,
    COUNT(DISTINCT t2.IdCliente) AS MAU,
    COUNT(DISTINCT t2.DtDay) AS DaysCount,
    SUM(t2.QtdePontos_pos) AS QtdePontos_pos,
    SUM(t2.QtdePontos_pos) / COUNT(DISTINCT t2.IdCliente) AS ARPU
FROM tb_distinct_days AS t1
LEFT JOIN tb_daily AS t2
    ON t2.DtDay <= t1.DtRef
    -- AND julianday(t1.DtRef) - julianday(t2.DtDay) < 28
    AND datediff(t1.DtRef, t2.DtDay) < 28
GROUP BY t1.DtRef
ORDER BY DtRef ASC 


Databricks visualization. Run in Databricks to view.